In [1]:
import pandas as pd
hourly = pd.read_parquet("hourly_report.parquet")

In [2]:
import pandas as pd
hourly = pd.read_parquet("hourly_report.parquet")

# Afficher le nombre de lignes où le taux d'utilisation est supérieur à 1
hourly["utilization_rate"] = hourly["flow_mw"] / hourly["capacity_mw"]

nans_utilization = hourly[hourly["utilization_rate"] > 1].shape[0]
print(f"Nombre de lignes avec taux d'utilisation > 1 : {nans_utilization}")

# Afficher le nombre de lignes où le taux d'utilisation est "inf"
nans_inf = hourly[hourly["utilization_rate"] == float("inf")].shape[0]
print(f"Nombre de lignes avec taux d'utilisation inf : {nans_inf}")

# Calculer le taux d'utilisation moyen quand le taux d'utilisation est > 1 mais pas inf
mean_utilization = hourly[(hourly["utilization_rate"] > 1) & (hourly["utilization_rate"] != float("inf"))]["utilization_rate"].mean()
print(f"Taux d'utilisation moyen pour les lignes avec taux > 1 mais pas inf : {mean_utilization}")

# Afficher par pays d'origine le nombre d'heures où le taux d'utilisation est supérieur à 1 mais pas inf

utilization_by_country = hourly[(hourly["utilization_rate"] > 1) & (hourly["utilization_rate"] != float("inf"))].groupby("from_country").size()
hours_by_country = hourly.groupby("from_country").size()
utilization_percentage = (utilization_by_country / hours_by_country) * 100
print("Pourcentage d'heures avec taux d'utilisation > 1 mais pas inf par pays :")
print(utilization_percentage)

# Taux d'utilisation moyen par pays d'origine pour un taux d'utilisation > 1 mais pas inf
mean_utilization_by_country = hourly[(hourly["utilization_rate"] > 1) & (hourly["utilization_rate"] != float("inf"))].groupby("from_country")["utilization_rate"].mean()
print("Taux d'utilisation moyen par pays d'origine pour les heures avec taux > 1 mais pas inf :")
print(mean_utilization_by_country)


Nombre de lignes avec taux d'utilisation > 1 : 126534
Nombre de lignes avec taux d'utilisation inf : 645
Taux d'utilisation moyen pour les lignes avec taux > 1 mais pas inf : 1.9044273991864242
Pourcentage d'heures avec taux d'utilisation > 1 mais pas inf par pays :
from_country
Belgium          21.664828
France           16.058140
Germany          26.993243
Great-Britain    10.438773
Italy-North       1.949822
Spain             0.600920
Switzerland       1.601187
dtype: float64
Taux d'utilisation moyen par pays d'origine pour les heures avec taux > 1 mais pas inf :
from_country
Belgium          2.340391
France           1.706289
Germany          2.379782
Great-Britain    1.768766
Italy-North      1.673726
Spain            1.323901
Switzerland      1.297857
Name: utilization_rate, dtype: float64


In [258]:
from src.load_files import *
from src.gen_reports import *
from src.visualization import *

flows_folder = "./data/flows"
price_folder = "./data/energy_prices"
capacities_folder = "./data/capacities"

In [231]:
flows = load_entsoe_folder(flows_folder, "flow")
prices = load_prices(price_folder)
capacities = load_entsoe_folder(capacities_folder, "capacity")
daily_capacities = load_daily_capacity(capacities_folder)


Lecture : Cross-border Physical Flows France 2021.csv
Lecture : Cross-border Physical Flows France 2022.csv
Lecture : Cross-border Physical Flows France 2023.csv
Lecture : Cross-border Physical Flows France 2024.csv
Lecture : Energy prices Belgium 2021.csv
Lecture : Energy prices Belgium 2022.csv
Lecture : Energy prices Belgium 2023.csv
Lecture : Energy prices Belgium 2024.csv
Lecture : Energy prices France 2021.csv
Lecture : Energy prices France 2022.csv
Lecture : Energy prices France 2023.csv
Lecture : Energy prices France 2024.csv
Lecture : Energy prices Germany 2021.csv
Lecture : Energy prices Germany 2022.csv
Lecture : Energy prices Germany 2023.csv
Lecture : Energy prices Germany 2024.csv
Lecture : Energy prices Italy-North 2021.csv
Lecture : Energy prices Italy-North 2022.csv
Lecture : Energy prices Italy-North 2023.csv
Lecture : Energy prices Italy-North 2024.csv
Lecture : Energy prices Spain 2021.csv
Lecture : Energy prices Spain 2022.csv
Lecture : Energy prices Spain 2023.csv

In [232]:
# calcul des flux monétaires
flows_money = compute_monetary_flows(flows, prices)

# merge avec les capacités horaires
hourly_report = flows_money.merge(
    capacities,
    on=["datetime","year","from_country","to_country","partner"],
    how="left",
    validate="one_to_one"
    )

In [247]:
df = hourly_report.copy()

# convertir datetime → jour (datetime64)
df["day"] = pd.to_datetime(df["datetime"]).dt.normalize()

# merge
df = df.merge(
    daily_capacities,
    left_on=["day", "from_country", "to_country"],
    right_on=["day", "from_country", "to_country"],
    how="left",
    suffixes=("", "_daily"),
    validate="many_to_one"  
)

# remplir uniquement les capacités manquantes
df["capacity_mw"] = df["capacity_mw"].fillna(df["capacity_mw_daily"])

# nettoyage
df = df.drop(columns=["day", "capacity_mw_daily"])

In [248]:
# calcul du taux d'utilisation
capacity_clean = df["capacity_mw"].replace(0, np.nan)

df["utilization_rate"] = np.where(
    capacity_clean.notna(),
    df["flow_mw"] / capacity_clean,
    np.nan
)

In [ ]:
def aggregate_yearly(df):
    """ Aggregate hourly report by year and partner, and compute statistics on flows, values, capacities and utilization rates.
    Input : dataframe with columns datetime, year, from_country, to_country, partner, flow_mw, price_export, price_import, value_eur, congestion_rent, capacity_mw, utilization_rate
    Output : dataframe with columns year, partner, export_twh, import_twh, export_value_M€, import_value_M€, export_price_avg_€/MWh, import_price_avg_€/MWh, capacity_coverage, export_capacity_mwh, import_capacity_mwh, export_utilization_rate, import_utilization_rate, export_abs_congestion_rent_M€, import_abs_congestion_rent_M€, utilization_rate, abs_congestion_rent_M€"""

    def compute_stats(g):

        export_flow = g.loc[g.from_country == "France", "flow_mw"].sum()
        import_flow = g.loc[g.to_country == "France", "flow_mw"].sum()

        export_value = g.loc[g.from_country == "France", "value_eur"].sum()
        import_value = g.loc[g.to_country == "France", "value_eur"].sum()

        export_abs_rent = g.loc[g.from_country == "France", "congestion_rent"].abs().sum()
        import_abs_rent = g.loc[g.to_country == "France", "congestion_rent"].abs().sum()

        export_price_avg = (export_value / export_flow) if export_flow > 0 else np.nan
        import_price_avg = (import_value / import_flow) if import_flow > 0 else np.nan

        total_abs_rent = export_abs_rent + import_abs_rent
        spread_avg = total_abs_rent / (export_flow + import_flow) if (export_flow + import_flow) > 0 else np.nan

        export_capacities = g.loc[g.from_country == "France", "capacity_mw"].sum()
        import_capacities = g.loc[g.to_country == "France", "capacity_mw"].sum()

        export_util = (g.loc[g.from_country == "France", "utilization_rate"].mean())
        import_util = (g.loc[g.to_country == "France", "utilization_rate"].mean())
        total_util_rate = (g["utilization_rate"].mean())

        # 3. OUTPUT
        return pd.Series({

            "export_twh": export_flow / 1e6,
            "import_twh": import_flow / 1e6,

            "export_value_M€": export_value / 1e6,
            "import_value_M€": import_value / 1e6,

            "import_price_avg_€/MWh": import_price_avg,
            "export_price_avg_€/MWh": export_price_avg,

            "export_abs_congestion_rent_M€": export_abs_rent / 1e6,
            "import_abs_congestion_rent_M€": import_abs_rent / 1e6,

            "export_capacity_twh": export_capacities / 1e6,
            "import_capacity_twh": import_capacities / 1e6,

            "export_utilization_rate": export_util,
            "import_utilization_rate": import_util,

            "utilization_rate": total_util_rate,
            "spread_avg_€/MWh": spread_avg,
            "abs_congestion_rent_M€": total_abs_rent / 1e6
        })

    yearly = (
        df.groupby(["year", "partner"])
        .apply(compute_stats)
        .reset_index()
        .rename(columns={"partner": "country"})
    )

    return yearly

In [270]:
yearly_report = aggregate_yearly(df)
df.drop(columns=["day"], inplace=True)
df.head()


,datetime,year,from_country,to_country,partner,flow_mw,price_export,price_import,value_eur,congestion_rent,capacity_mw,utilization_rate
0,2021-01-01,2021,Belgium,France,Belgium,476.68,50.87,50.87,24248.7116,0.0,700.0,0.680971
1,2021-01-01,2021,France,Belgium,Belgium,0.00,50.87,50.87,0.0000,0.0,1800.0,0.000000
2,2021-01-01,2021,France,Germany,Germany,0.00,50.87,50.87,0.0000,0.0,1800.0,0.000000
3,2021-01-01,2021,France,Great Britain,Great Britain,2019.00,50.87,NaN,102706.5300,NaN,2000.0,1.009500
4,2021-01-01,2021,France,Italy-North,Italy-North,371.00,50.87,50.87,18872.7700,0.0,2400.0,0.154583


In [277]:

def load_daily(folder_path):
    """ Charge files with daily capacities and merge them to extract relevant info (zones, date, capacity)"""
    
    # liste des fichiers
    path = folder_path + "/daily/*.csv"
    years = [str(y) for y in range(2021, 2025)]
    files = glob.glob(path)

    # lecture + concaténation
    df = pd.concat(
        (pd.read_csv(f) for f in files if any(year in f for year in years)),
        ignore_index=True
    )

    # enlever n/e et renommer en capacity_mw
    df["Capacity (MW)"] = pd.to_numeric(
        df["Capacity (MW)"],
        errors="coerce"
    )
    df = df.rename(columns={"Capacity (MW)": "capacity_mw"})

    # extraction zones robuste
    df["from_country"] = (
        df["Out Area"]
        .str.extract(r"\|(.*)")
        .iloc[:, 0]
        .map(zone_map)
    )

    df["to_country"] = (
        df["In Area"]
        .str.extract(r"\|(.*)")
        .iloc[:, 0]
        .map(zone_map)
    )

    # Supprimer la colonne "Time Interval"
    df = df.drop(columns=["Time Interval"])

    # Mettre au format datetime
    df["day"] = pd.to_datetime(
        df["Day"].str.split(" ").str[1],
        format="%d/%m/%Y"
    )

    # Agréger par jour et par paire de pays en prenant la somme des capacités
    df = (
        df.groupby(["day","from_country","to_country"], as_index=False)
        ["capacity_mw"]
        .sum()
    )

    return df[[
        "day",
        "from_country",
        "to_country",
        "capacity_mw"
    ]]

In [ ]:
# import des fonctions des autres fichiers
from src.load_files import *

# flows_folder = "./data/flows"
# price_folder = "./data/energy_prices"
# capacities_folder = "./data/capacities"

# flows = load_entsoe_folder(flows_folder, "flow")
# prices = load_prices(price_folder)
# capacities = load_entsoe_folder(capacities_folder, "capacity")
daily_capacities = load_daily_capacity(capacities_folder)
daily_cap = load_daily(capacities_folder)
duplicates = daily_cap.duplicated(subset=["day", "from_country", "to_country"]).sum()
print(f"Nombre de doublons dans daily_cap : {duplicates}")
print(f"Nombre de doublons dans daily_capacities : {daily_capacities.duplicated(subset=['day', 'from_country', 'to_country']).sum()}")

# hourly_report1 = merge_hourly_report(flows, prices, capacities, daily_capacities)


Nombre de doublons dans daily_cap : 0
Nombre de doublaons dans daily_capacities : 14650


In [267]:
# calcul du taux d'utilisation moyen horaire pour Belgique et Germany dans hourly_report en 2021
belgium_utilization = df[(df["partner"] == "Belgium") & (df["year"] == 2021)]["utilization_rate"].mean()
germany_utilization = df[(df["partner"] == "Germany") & (df["year"] == 2021)]["utilization_rate"].mean()
spain_utilization = df[(df["partner"] == "Spain") & (df["year"] == 2021)]["utilization_rate"].mean()
italy_utilization = df[(df["partner"] == "Italy-North") & (df["year"] == 2021)]["utilization_rate"].mean()
switzerland_utilization = df[(df["partner"] == "Switzerland") & (df["year"] == 2021)]["utilization_rate"].mean()
uk_utilization = df[(df["partner"] == "Great Britain") & (df["year"] == 2021)]["utilization_rate"].mean()

print(f"Taux d'utilisation moyen pour la Belgique : {belgium_utilization}")
print(f"Taux d'utilisation moyen pour l'Allemagne : {germany_utilization}")
print(f"Taux d'utilisation moyen pour le Royaume-Uni : {uk_utilization}")
print(f"Taux d'utilisation moyen pour l'Italie : {italy_utilization}")
print(f"Taux d'utilisation moyen pour l'Espagne : {spain_utilization}")
print(f"Taux d'utilisation moyen pour la Suisse : {switzerland_utilization}")

Taux d'utilisation moyen pour la Belgique : 0.7467581427388147
Taux d'utilisation moyen pour l'Allemagne : 0.4891945136459969
Taux d'utilisation moyen pour le Royaume-Uni : 0.38793588378587907
Taux d'utilisation moyen pour l'Italie : 0.3762799950577858
Taux d'utilisation moyen pour l'Espagne : 0.37551087767363284
Taux d'utilisation moyen pour la Suisse : 0.2537429148872847


In [252]:
# Taux d'utilisation journalier pour l'Espagne avec une imputation par la moyenne journalière de chaque heure
df2 = df.copy()

# convertir datetime → jour (datetime64)
df2["day"] = pd.to_datetime(df2["datetime"]).dt.normalize()

# merge
df2 = df2.merge(
    daily_capacities,
    left_on=["day", "from_country", "to_country"],
    right_on=["day", "from_country", "to_country"],
    how="left",
    suffixes=("", "_daily"),
    validate="many_to_one"  
)

# calcul du taux d'utilisation journalier
capacity_clean = df2["capacity_mw_daily"].replace(0, np.nan)

df2["utilization_rate_daily"] = np.where(
    capacity_clean.notna(),
    df2["flow_mw"] / capacity_clean,
    np.nan
)

In [255]:
# calcul du taux d'utilisation moyen horaire pour Belgique et Germany dans hourly_report
belgium_utilization = df2[df2["partner"] == "Belgium"]["utilization_rate_daily"].mean()
germany_utilization = df2[df2["partner"] == "Germany"]["utilization_rate_daily"].mean()
spain_utilization = df2[df2["partner"] == "Spain"]["utilization_rate_daily"].mean()
italy_utilization = df2[df2["partner"] == "Italy-North"]["utilization_rate_daily"].mean()
switzerland_utilization = df2[df2["partner"] == "Switzerland"]["utilization_rate_daily"].mean()
uk_utilization = df2[df2["partner"] == "Great Britain"]["utilization_rate_daily"].mean()

print(f"Taux d'utilisation moyen pour la Belgique : {belgium_utilization}")
print(f"Taux d'utilisation moyen pour l'Allemagne : {germany_utilization}")
print(f"Taux d'utilisation moyen pour l'Espagne : {spain_utilization}")
print(f"Taux d'utilisation moyen pour l'Italie : {italy_utilization}")
print(f"Taux d'utilisation moyen pour la Suisse : {switzerland_utilization}")
print(f"Taux d'utilisation moyen pour le Royaume-Uni : {uk_utilization}")

Taux d'utilisation moyen pour la Belgique : 0.6533506381758942
Taux d'utilisation moyen pour l'Allemagne : 0.556962748698782
Taux d'utilisation moyen pour l'Espagne : 0.4908074892024967
Taux d'utilisation moyen pour l'Italie : 0.4440092530925744
Taux d'utilisation moyen pour la Suisse : 0.30030226379484243
Taux d'utilisation moyen pour le Royaume-Uni : 0.18271822841546503
